# Motifs in a spiking circuit

Build a circuit JSON (the same file the Godot game and the C++ engine read) in PyNEST, run it, then look at its 3-node motifs against degree-preserving nulls, and at spike statistics.

PyNEST must be importable: add the NEST build's `site-packages` to `PYTHONPATH` (for example `third_party/nest-install/lib/python3.*/site-packages`) before starting Jupyter.

In [1]:
import sys, glob, os
for d in glob.glob('../../third_party/nest-install/lib/python3*/site-packages') + glob.glob('/tmp/nest-install/lib/python3*/site-packages'):
    sys.path.append(d)
sys.path.insert(0, '..')
import numpy as np, matplotlib.pyplot as plt
import spike

## Build and run

`spike.build` creates populations, devices and connections in the same order as the C++ engine, so the same seed gives identical spikes (checked by `tests/test_parity.py`).

In [2]:
net = spike.build('ei_net.json')
spikes = spike.run(net, 1000.0)
print(net.n_neurons, 'neurons,', len(spikes), 'spikes')
plt.figure(figsize=(8, 3)); plt.plot(spikes.t_ms, spikes.neuron, '|', ms=3, color='k')
plt.axhline(79.5, color='tab:cyan', lw=.8); plt.xlabel('t (ms)'); plt.ylabel('neuron (E below the line, I above)'); plt.show()


             -- N E S T --

 Copyright (C) 2004 The NEST Initiative

 Version: 3.10.0-post0.dev0
 Built  : Nov 15 2026 13:34:39

 This program is provided AS IS and comes with NO WARRANTY.
 See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.

100 neurons, 3827 spikes


## Spike statistics

In [3]:
T = 1000.0
rates = spike.stats.firing_rates(spikes, net.n_neurons, 0, T)
cv = spike.stats.isi_cv(spikes, net.n_neurons, 0, T)
print('mean rate E %.1f Hz, I %.1f Hz' % (rates[:80].mean(), rates[80:].mean()))
print('ISI CV (median over neurons with >=3 spikes): %.2f' % np.nanmedian(cv))
print('Golomb chi (1 ms bins): %.3f' % spike.stats.golomb_chi(spikes, net.n_neurons, 0, T, 1.0))
h = spike.stats.psth(spikes, net.n_neurons, None, 0, T, 10.0)
plt.figure(figsize=(8, 2.5)); plt.bar(np.arange(len(h)) * 10 + 5, h, width=10); plt.xlabel('t (ms)'); plt.ylabel('rate (Hz/neuron)'); plt.show()

mean rate E 38.2 Hz, I 38.6 Hz
ISI CV (median over neurons with >=3 spikes): 0.48
Golomb chi (1 ms bins): 0.122


## Motif census against degree-preserving nulls

The edge list comes from NEST `GetConnections`. Milo et al. 2002 motif ids: 38 is the feed-forward loop, 98 the feedback loop. A z-score says how many standard deviations the real count is from rewired graphs that keep every neuron's in- and out-degree (`seed` makes the nulls reproducible; the C++ library gives the same numbers).

In [4]:
edges = spike.motifs.edges_from_nest(net)
res = spike.motifs.zscores(edges, net.n_neurons, n_null=100, seed=1)
for idx, milo, man, name, count, mu, sd, z in res.table():
    print(f'{milo:>4} {man:>5} {name:<24} count={count:>6}  null={mu:>9.1f} +- {sd:>6.1f}  z={z:>6.2f}')

   6  021D fan_out                  count=  4661  null=   4398.7 +-  133.0  z=  1.97
  12  021C chain                    count=  8308  null=   7740.9 +-  253.1  z=  2.24
  14  111U mutual_pair_driving      count=   827  null=   1141.4 +-  130.3  z= -2.41
  36  021U fan_in                   count=  3715  null=   3500.8 +-  110.4  z=  1.94
  38  030T feed_forward_loop        count=  1201  null=   1027.9 +-   57.5  z=  3.01
  46  120U mutual_pair_co_driving   count=    63  null=     75.5 +-    9.8  z= -1.27
  74  111D mutual_pair_fed          count=   641  null=    909.4 +-  107.2  z= -2.50
  78   201 two_mutual_pairs         count=    31  null=     69.1 +-   17.7  z= -2.15
  98  030C feedback_loop            count=   378  null=    333.0 +-   23.4  z=  1.93
 102  120C mutual_pair_in_loop      count=   100  null=    134.6 +-   16.3  z= -2.13
 108  120D mutual_pair_co_driven    count=    50  null=     60.3 +-    8.1  z= -1.28
 110   210 mutual_pair_double       count=    12  null=     17.9 

In [5]:
z = np.array(res.z); z = np.clip(np.nan_to_num(z, posinf=10, neginf=-10), -10, 10)
plt.figure(figsize=(8, 3)); plt.bar([str(m.milo_id) for m in spike.motifs.MOTIFS], z); plt.axhline(0, color='k', lw=.5)
plt.xlabel('Milo motif id'); plt.ylabel('z-score'); plt.show()

## Excitatory or inhibitory sub-graph only

`sign='E'` keeps edges with positive weight, `sign='I'` negative.

In [6]:
for s in ('E', 'I'):
    r = spike.motifs.zscores(edges, net.n_neurons, sign=s, n_null=50, seed=1)
    top = max(r.table(), key=lambda row: abs(row[7]) if np.isfinite(row[7]) else 0)
    print(s, 'edges:', spike.motifs.make_graph(edges, net.n_neurons, s).edge_count(), '| most over/under-represented:', top[3], 'z=%.2f' % top[7])

E edges: 761 | most over/under-represented: feedback_loop z=1.39


I edges: 371 | most over/under-represented: fan_in z=-1.65
